In [1]:
import pandas as pd
import pathlib
import PyPDF2

year = 2026

## Step 1: Download Data from Conftool

### 1a: Download PDF files
In ConfTool, filter to final set of papers using paper status, and make sure to use the `Anonymous format with contribution ID only` option for file names.
https://www.conftool.pro/simbuild2026/index.php?page=adminPapersBrowse&filter=hide&form_tracks=3&form_format=-&form_topic=-&form_status=a&specialfilter=-&form_withdrawn=no&version=final&form_nameformat=anonymous
Then select `Save all 1st Files of Current Page as ZIP` 


### 1b: Download List of Contributions
`Export Contributions` from ConfTool, and select `Include abstracts of submissions`
https://www.conftool.pro/simbuild2026/index.php?page=adminExport&export_select=papers


## Step 2: Convert to CSV
May have to upload to Google Drive and download as CSV if Excel doesn't cooperate

## Step 3: Load Data and Add New Columns


In [2]:
def getPageCount(id):
    fileName = 'simbuild{}final/Contribution_{}_final_a.pdf'.format(year, id)
    fileURL = 'https://publications.ibpsa.org/proceedings/simbuild/{}/papers/{}'.format(year, fileName)
    pdfReader = PyPDF2.PdfReader(fileName)
    pageCount = len(pdfReader.pages)
    return fileURL, pageCount


df = pd.read_csv('SimBuild{}.csv'.format(year))

df.rename(columns={'paperID': 'paper_id'}, inplace=True)

tmp = df.apply(lambda row: getPageCount(row['paper_id']), axis=1, result_type='expand')
df['filename'] = tmp[0]
df['number_of_pages'] = tmp[1]
df['conference_paper_id'] = df.apply(lambda row: '{}_{}'.format(row['conference_id'], row['paper_id']), axis=1, result_type='expand')
df['page_end'] = df['number_of_pages'].cumsum()
df['page_start'] = df['page_end'] - df['number_of_pages'] + 1

finalCols = [   'conference_paper_id', 'paper_id', 'filename', 'page_start', 'page_end', 'number_of_pages', 'DOI', 'topic_id', 
                'authors', 'organisations', 'title', 'keywords', 'abstract', 'conference_id']
df = df[finalCols]

df.head()


df.to_csv('SimBuild{}_formatted.csv'.format(year), index=False)